# Lab 04 · One configured NDVI job

Automated Workflows in Satellite Image Processing · Version 1.0 · 26 October 2026

In Lab 3 you acquired small red and NIR subsets and kept enough records to explain
where they came from. Today you turn those files into one repeatable processing
job. A later workflow manager will call this same job for many scenes.

The calculation is short. Most of the work is deciding what the calculation means,
checking its inputs, and keeping an honest account when execution fails.

You will write the database extension, repair three small functions, then run the
supplied CLI through success, reuse and recovery. Keep this notebook beside its
companion files. Start with README.md if you need to add packages.


In [ ]:
from pathlib import Path
import json
import sqlite3
import subprocess
import sys
import numpy as np
import rasterio
import matplotlib.pyplot as plt

ROOT = Path.cwd()
assert (ROOT / "lab04_cli.py").is_file(), "Open this notebook from the extracted Lab 4 folder."

def cli(*arguments, expected=(0,)):
    result = subprocess.run([sys.executable, "lab04_cli.py", *map(str, arguments)], capture_output=True, text=True)
    if result.returncode not in expected:
        raise RuntimeError(f"Exit {result.returncode}: {result.stdout} {result.stderr}")
    return json.loads(result.stdout)

def check(part):
    result = subprocess.run([sys.executable, "check_lab04.py", "--part", part], capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    return result.returncode == 0

def query(config, sql, params=()):
    import yaml
    config = Path(config).resolve()
    settings = yaml.safe_load(config.read_text())
    database = (config.parent / settings["database"]).resolve()
    with sqlite3.connect(database.as_uri() + "?mode=ro", uri=True) as connection:
        connection.row_factory = sqlite3.Row
        connection.execute("PRAGMA foreign_keys=ON")
        return [dict(row) for row in connection.execute(sql, params)]

print("Python:", sys.version.split()[0], "Rasterio:", rasterio.__version__)

## Exercise 1 · Write the processing schema

Write all six tables in student_processing_schema.sql using SCHEMA_CONTRACT.md.
You already designed the acquisition tables in Lab 3; support/ contains those
completed tables only. You are designing the processing extension here.

Start by explaining the difference between these records:

- A request to compute NDVI from two exact artifacts under specified settings.
- An attempt that could succeed, fail, stop unexpectedly or reuse a result.
- A file artifact whose bytes and interpretation are identified.
- A current location where those bytes might or might not still exist.

Lab 3 requires every artifacts row to reference an acquisition subset. NDVI is a
derived product. The processing registry links acquired inputs back to that table
and stores derived products without inventing acquisition records.

Decide which keys allow two retries of one request, while preventing two red
inputs in the same role. Decide which failures a foreign key can detect, and which
require checking files. Keep the existing discovery runs unchanged.

Run the check whenever you want feedback. Its failures are expected until your
schema is complete.

In [ ]:
schema_ok = check("schema")

Explain one constraint that prevents bad lineage, and one condition that your
schema cannot establish by itself.

*Write here.*

## Exercise 2 · What makes this the same request?

Repair make_request_spec in repair_functions.py. It currently uses dictionary
iteration order, omits important evidence and includes a storage path.

Return (normalized_spec, fingerprint). The normalized spec must be JSON compatible
and contain the stage, normalized scientific settings, ordered red/NIR input
records, environment and the policy name exact_environment_v1.

Keep all scientific settings, including a calibration review when present. Use
each input's artifact_id, sha256, size_bytes, raster and lineage; keep role names
explicit and in red, then nir order. Exclude registration times and runtime paths.
Normalize numeric parameters so 1 and 1.0 have the same interpretation. Produce
SHA-256 from JSON with sorted keys, compact separators and no NaN values.

This lab uses a conservative equivalence policy: a code or environment change
creates a new request. A later policy could permit equivalence across tested
environments, but that requires an explicit decision.

Predict the result of each change before running the check:

| Change | Same request or different request? Why? |
| --- | --- |
| Write to a different disk | |
| Reorder YAML keys | |
| Retry after a missing file is restored | |
| Change the AOI or input checksum | |
| Change the denominator threshold | |
| Repair the algorithm implementation | |
| Change the calibration interpretation | |

In [ ]:
identity_ok = check("identity")

## Exercise 3 · A program can finish and still be wrong

Repair validate_pair and calculate_ndvi in repair_functions.py. These are logical
repairs: the supplied functions execute, but their assumptions are incomplete.

validate_pair must reject different scenes, collections, AOIs or extraction
recipes; incorrect band roles; and incompatible CRS, shape or pixel transforms.
Check that required radiometric metadata are finite and the scale is positive.
Equal shapes do not establish matching ground positions. This stage requires a
matching grid; resampling would need its own explicit preparation policy.

calculate_ndvi receives the stored DN arrays and the effective calibration chosen
in the request. Mask original nodata, convert before subtraction or addition, then
apply scale and offset separately for each band. Use
NDVI = (NIR reflectance - red reflectance) / (NIR reflectance + red reflectance).
Return float32, put NaN at invalid pixels and at denominators with absolute value
at most epsilon, and leave the inputs unchanged. Do not clip the saved values.

The calculation must support nonzero offsets. The demo's reviewed offset 0 is not
a reason to hard-code zero. The checks use the real arrays with their original
metadata as well as a deliberately high denominator threshold.

In [ ]:
pair_ok = check("pair")
ndvi_ok = check("ndvi")

Explain these two bugs with a concrete example from the metadata or arrays:

1. Why can uint16 subtraction give the wrong sign?
2. Why does a shared multiplicative factor cancel in NDVI, while a shared additive
   offset generally does not?

*Write here.*

## Run one complete job

Finish the repairs before proceeding. The next cell runs a fresh Python process,
so it uses your current saved files rather than a notebook's cached module.

Use your own completed Lab 3 catalogue through the init/inputs/configure commands
in README.md, then change CONFIG below. Or leave the default to import the small
preserved real example. This explicit local route works without network access.
All following main-run cells use the CONFIG you choose.

In [ ]:
assert check("all"), "Finish the schema and function repairs before creating processing evidence."

CONFIG = Path("demo/config/ndvi.yml")
if CONFIG == Path("demo/config/ndvi.yml") and not CONFIG.exists():
    print(cli("demo"))
assert CONFIG.is_file(), "Create your configuration using the README commands first."

import yaml
print(CONFIG.read_text())

Inspect the configuration before running. Which artifact fills each role? Where
will the output persist? What calibration does it request? What does nodata_only
say about clouds?

The plan command resolves identity and prints the normalized request without
registering a request or creating an attempt.

In [ ]:
plan = cli("plan", "--config", CONFIG)
print("Planned request:", plan["request_id"])
print(json.dumps(plan["normalized_spec"]["settings"], indent=2))
first = cli("run", "--config", CONFIG)
print(json.dumps(first, indent=2))
assert first["request_id"] == plan["request_id"]

Predict what the next invocation changes in the database. Then run it. If you
have already run the notebook, the first result above may itself be a reuse.

In [ ]:
repeat = cli("run", "--config", CONFIG)
print(json.dumps(repeat, indent=2))
assert repeat["request_id"] == first["request_id"]
assert repeat["attempt_id"] != first["attempt_id"]
assert repeat["status"] == "reused"

### Inspect the raster and its record

The display below uses a fixed NDVI color range. That only changes the display;
it does not clip the stored raster. Look at the reported coverage and range too.
No cloud classification data have been used.

In [ ]:
output_path = Path(repeat["path"])
from matplotlib.ticker import MaxNLocator
with rasterio.open(output_path) as source:
    ndvi = source.read(1, masked=True)
    bounds = source.bounds
    print("Provenance tags:", source.tags())
    fig, ax = plt.subplots(figsize=(7, 6))
    image = ax.imshow(ndvi, cmap="RdYlGn", vmin=-1, vmax=1,
                      extent=(bounds.left, bounds.right, bounds.bottom, bounds.top))
    ax.set(title="AOI NDVI · nodata only; clouds not screened", xlabel="Easting (m)", ylabel="Northing (m)")
    ax.xaxis.set_major_locator(MaxNLocator(4))
    fig.colorbar(image, ax=ax, label="NDVI")
    plt.show()

In [ ]:
history = query(CONFIG, """SELECT attempt_id,request_id,status,stage,started_at,finished_at,
    error_code,reused_from_attempt_id FROM processing_attempts ORDER BY started_at""")
history

Write a query in investigation_queries.sql that starts with the NDVI artifact
and traces it to its successful producing attempt, request, input roles and the
original Lab 3 scene/asset/AOI records. Run your query below. The output-to-input
links are the explanation; filenames are only convenient locations.

Replace the harmless starter query with your lineage query.

In [ ]:
query(CONFIG, "SELECT artifact_id, sha256, size_bytes FROM processing_artifacts")

What is the difference between producing an artifact and returning it from a
reused attempt? Where is the calibration review preserved?

*Write here.*

## Exercise 4 · Diagnose before retrying

These cases use separate copies of the same real example under cases/. They leave
your main archive alone and refuse to overwrite an existing case. The helper below
lets you inspect a case again when you rerun this notebook.

### Case A · A recorded input has disappeared

Predict the new attempt's status and the location status. The case moves its own
red file aside, then calls the normal job. It does not change the request settings.
The deliberate failure exit code is 1.

In [ ]:
def ensure_case(name):
    config = Path("cases") / name / "config/ndvi.yml"
    if not config.exists():
        result = cli("case", "--name", name, expected=(1, 86))
        print("Created case:", name, "child exit:", result["exit_code"])
    return config

missing_config = ensure_case("missing-input")
query(missing_config, "SELECT attempt_id,request_id,status,stage,error_code,error_message FROM processing_attempts")

In [ ]:
query(missing_config, "SELECT artifact_id,local_path,status,checked_at FROM processing_artifact_locations")

Add diagnostic SQL to investigation_queries.sql. Identify the exact input role
that failed and explain why the old input artifact still exists in the database.
Predict whether restoring the identical file creates a new scientific request.

*Write here.*

After recording your diagnosis, restore the case input and retry. Do not delete
the failed attempt.

In [ ]:
case_root = missing_config.parent.parent
note = json.loads((case_root / "case_note.json").read_text())
if Path(note["held"]).exists() and not Path(note["original"]).exists():
    print(cli("restore-input", "--config", missing_config))
recovered_missing = cli("run", "--config", missing_config)
print(json.dumps(recovered_missing, indent=2))
query(missing_config, "SELECT request_id,status,error_code FROM processing_attempts ORDER BY started_at")

### Case B · The process stops before publication

The case writes and validates a real temporary NDVI file, commits a receipt, then
the child process immediately exits with code 86. It cannot run an exception
handler. The parent command waits for that child to stop.

Before running it, predict: which status will the database show, which file will
exist, and will a processing_outputs row exist?

In [ ]:
stopped_config = ensure_case("before-rename")
stopped_rows = query(stopped_config, "SELECT attempt_id,request_id,status,stage,temp_path,final_path,receipt_json FROM processing_attempts ORDER BY started_at")
for row in stopped_rows:
    print({key: row[key] for key in ("attempt_id", "status", "stage")},
          "temp exists:", Path(row["temp_path"]).exists(), "final exists:", Path(row["final_path"]).exists())
print("Output links:", query(stopped_config, "SELECT * FROM processing_outputs"))

A running row does not prove a process is still running. A missing error does
not prove success. Write down the evidence establishing that this child stopped.
Explain why retrying an unresolved running attempt is blocked in this lab.

*Write here.*

After inspecting the evidence, reconcile the exact stopped attempt. The assertion
--confirm-stopped is justified here by the parent waiting for the child. On a
cluster you would need scheduler/process evidence before making that assertion.

In [ ]:
running = [row for row in stopped_rows if row["status"] == "running"]
for row in running:
    print(cli("reconcile", "--config", stopped_config, "--attempt-id", row["attempt_id"], "--confirm-stopped"))
recovered_stop = cli("run", "--config", stopped_config)
print(json.dumps(recovered_stop, indent=2))
query(stopped_config, "SELECT request_id,attempt_id,status,stage,error_code FROM processing_attempts ORDER BY started_at")

Why is this a new attempt of the same request? What happened to the temporary
file? Which SQL rows show that recovery did not erase the interruption?

*Write here.*

## Optional · The file was renamed, but the database transaction never happened

Generate `python lab04_cli.py case --name after-rename`, inspect its paths,
receipt and absent output link, then reconcile it using the exact attempt ID.
The same reconciliation command validates the final file and adopts it. Explain
why recomputing is unnecessary and why file existence alone would be insufficient.

The rename and database commit are separate operations. A transaction in SQLite
does not make a filesystem rename part of that transaction.

Further investigation: move an output aside, run the main job again and explain
why the historical succeeded row remains while availability changes. Use a copy
of your workspace and retain the file so you can restore it.

*Write your evidence here if you complete an extension.*

## Evidence to hand in

Submit student_processing_schema.sql, repair_functions.py, this completed
notebook, investigation_queries.sql and small JSON reports for the main job and
both required cases. Keep actual data, rasters, databases and logs locally for
demonstration, rather than adding them to Git.

The next cell saves reports. It does not recreate an image or recompute NDVI.

In [ ]:
for label, config in (("main", CONFIG), ("missing_input", missing_config), ("interrupted", stopped_config)):
    destination = Path("evidence") / (label + "_report.json")
    cli("report", "--config", config, "--out", destination)
    print(destination)

Finish with a short explanation of what a future coordinator needs from this
job: how to specify the intended calculation, recognize a usable result and decide
whether to retry. Which part still needs a human scientific judgment?

*Write here.*

Your final evidence should show one complete NDVI lineage, a reuse invocation,
a preserved missing-input failure followed by recovery, and a stopped attempt
followed by reconciliation and a successful retry. Restart the kernel and run the
notebook again after saving your repairs; the notebook inspects existing case
workspaces and appends valid main invocations without overwriting the evidence.